# 시뮬

In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats, integrate, optimize
from dataclasses import dataclass
from typing import Callable

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. RNG (난수 생성기) 독립 스트림

**rng란?** 난수를 차례대로 뽑아주는 객체. 같은 seed로 만들면 언제 실행해도 같은 순서의 값이 나오므로(재현 가능), 실험을 매번 똑같이 돌려볼 수 있음.

**왜 하나의 rng를 계속 재사용하면 안 되는가**: 예를 들어 training 데이터와 calibration 데이터를 같은 rng로 순차적으로 만들면, training 쪽 코드를 조금만 고쳐도(난수를 하나라도 더 뽑거나 덜 뽑거나) calibration 데이터까지 달라져버림.

**해결책**: `substream(scenario, b, stage)`처럼 이름표(path)를 붙여서, 이름표가 다르면 서로 다른 재현 가능한 난수 stream을 만드는 함수를 만듦. 같은 이름표 → 같은 결과(재현), 다른 이름표 → 서로 다른 난수열.

문자열을 숫자로 바꿀 때는 파이썬 내장 `hash()`를 쓰지 않음(실행마다 값이 바뀌는 경우가 있어 재현성이 깨짐) — 대신 `hashlib.sha256`으로 실행 환경과 무관하게 항상 고정된 수로 바꿈 (CLAUDE.md §9).

In [2]:
import hashlib

# 이 값이 같으면 전체 실험이 항상 또같이 재현됨
MASTER_SEED = 20240911


def stable_int(value):
    """문자열/정수를 항상 같은 정수로 바꿀 (파이썬 hash()는 실행마다 바뀌어서 사용 금지)."""
    text = str(value)
    digest = hashlib.sha256(text.encode("utf-8")).digest()  # 문자열 -> 32바이트 고정 지문값
    return int.from_bytes(digest[:8], "little")  # 그 중 8바이트만 정수로 변환


def substream(*path):
    """(scenario, b, r, stage, ...) 같은 이름표로 독립적인 rng를 생성.

    같은 path -> 항상 같은 난수열 (재현 가능)
    다른 path -> 서로 다른 재현 가능한 난수열
    """
    entropy = [MASTER_SEED] + [stable_int(p) for p in path]
    seed_sequence = np.random.SeedSequence(entropy)
    return np.random.default_rng(seed_sequence)
# 간단한 확인: 같은 path는 같은 값, 다른 path(시나리오/stage)는 다른 값
rng_a = substream("linear_homo_gaussian", 1, "train")
rng_b = substream("linear_homo_gaussian", 1, "train")
rng_c = substream("linear_homo_gaussian", 1, "model")  # stage만 다름

values_a = rng_a.standard_normal(3)
values_b = rng_b.standard_normal(3)
values_c = rng_c.standard_normal(3)

print("rng_a :", values_a)
print("rng_b :", values_b)
print("rng_c :", values_c)

assert np.array_equal(values_a, values_b)  # 같은 path -> 같은 난수열
assert not np.array_equal(values_a, values_c)  # 다른 path -> 다른 난수열
print("RNG 기본 성질 확인 완료")

rng_a : [-0.578851  0.592685  1.246618]
rng_b : [-0.578851  0.592685  1.246618]
rng_c : [0.064885 2.014959 1.352782]
RNG 기본 성질 확인 완료


## 2. Error distribution

각 error는 `sample(size, rng)`, `cdf(e)`, `ppf(p)` 세 메서드를 갖는 클래스. 이 세 메서드가 같은 표준화 상수(df, scale, a, s 등)를 공유해야 해서 함수 대신 클래스로 묶음.

- `sample(size, rng)`: 실제 난수 표본을 size개 뽑음
- `cdf(e)`: $P(\varepsilon \le e)$. coverage 계산에 쓰임
- `ppf(p)`: cdf의 역함수(분위수). 예를 들어 `ppf(0.975)`는 상위 2.5% 지점 값

세 분포 모두 $E[\varepsilon]=0$, $\mathrm{Var}(\varepsilon)=1$로 표준화됨:
- Gaussian: 정규분포
- Student-$t$: 대칭이지만 꼬리가 두꺼운 분포
- Lognormal: 비대칭 분포

In [ ]:
class GaussianError:
    """E[eps]=0, Var(eps)=1인 표준정규 오차. 표준정규는 원래부터 평균0·분산1이라 별도 변환이 필요 없음."""

    def sample(self, size, rng):
        # 실제 난수를 size개 생성 (몬테카를로 표본, training/calibration의 잡음 등에 사용)
        return rng.standard_normal(size)

    def cdf(self, e):
        # cumulative distribution function: P(eps <= e). coverage 계산에 사용
        return stats.norm.cdf(e)

    def ppf(self, p):
        # cdf의 역함수(quantile function): P(eps <= q) = p 인 q를 반환
        # 예: ppf(0.975) -> 상위 2.5% 지점 값. Oracle 구간의 upper bound 계산에 사용
        return stats.norm.ppf(p)


class StudentTError:
    """T ~ t_df를 sqrt(df/(df-2))로 나누어 평균 0·분산 1로 표준화. 기본값은 df=3."""

    def __init__(self, df = 3):
        if df <= 2:
            raise ValueError("분산을 1로 표준화하려면 df > 2 이어야 함")
        self.df = df  # 자유도(degrees of freedom); 작을수록 꼬리가 두꺼움(heavy tail)
        # t_df 분포의 분산은 df/(df-2). 이 값으로 나눠주면 최종 eps의 분산이 정확히 1이 됨
        self.scale = np.sqrt(df / (df - 2))

    def sample(self, size, rng):
        raw_t_sample = rng.standard_t(self.df, size = size)  # 아직 분산이 1이 아닌 원래 t분포 표본
        return raw_t_sample / self.scale  # scale로 나눠서 분산을 1로 맞춤

    def cdf(self, e):
        # 우리 eps는 원래 t분포를 scale로 나눈 것이므로,
        # 반대로 e에 scale을 다시 곱하면 원래 t분포 값으로 되돌아감 -> 거기서 t분포의 cdf를 사용
        e_in_original_t_scale = np.asarray(e) * self.scale
        return stats.t.cdf(e_in_original_t_scale, df=self.df)

    def ppf(self, p):
        q_in_original_t_scale = stats.t.ppf(p, df=self.df)  # 원래 t분포 기준의 분위수
        return q_in_original_t_scale / self.scale  # 우리 ε 기준(분산 1)으로 다시 변환


class LogNormalError:
    """eps = (exp(Z) - a) / s, Z~N(0,1), a=exp(1/2), s=sqrt(e*(e-1)).

    exp(Z) 자체는 평균이 0도 아니고 분산도 1이 아니므로(로그정규분포),
    평균을 a만큼 빼고 표준편차 s로 나눠서 최종 eps의 평균 0·분산 1을 맞춤.
    """

    def __init__(self):
        self.a = np.exp(0.5)  # exp(Z)의 평균 (Z~N(0,1)일 때 E[exp(Z)] = exp(1/2))
        self.s = np.sqrt(np.e * (np.e - 1))  # exp(Z)의 표준편차

    def sample(self, size, rng):
        z = rng.standard_normal(size)  # 먼저 표준정규 Z를 뽑고
        raw_lognormal = np.exp(z)  # Z를 exp에 넣으면 로그정규분포가 됨 (평균 0·분산 1 아님)
        return (raw_lognormal - self.a) / self.s  # 평균 0·분산 1로 표준화

    def cdf(self, u):
        # 목표: eps <= u 일 확률을 구하는 것
        # 1단계: 우리 eps 기준값 u를, 표준화하기 전의 "원래 exp(Z)" 기준값으로 되돌림
        #        eps = (exp(Z)-a)/s 였으니 반대로 풀면 exp(Z) = s*u + a
        u = np.asarray(u, dtype=float)
        original_scale_value = self.s * u + self.a  # 이게 바로 "s*u+a" = 원래 exp(Z) 기준값

        # 2단계: exp(Z) <= original_scale_value 일 확률을 구하면 됨.
        #        exp(Z)는 항상 양수이므로, original_scale_value <= 0이면 그 확률은 무조건 0
        is_in_support = original_scale_value > 0

        # 3단계: original_scale_value가 0보다 클 때만 log를 취해도 안전함.
        #        0 이하인 곳은 나중에 버릴 값이지만, log(0 이하)는 계산 자체가 에러/경고를 내므로
        #        일단 안전한 더미값 1.0으로 채워서 계산만 통과시킴 (결과는 다음 줄에서 덮어씀)
        safe_value_for_log = np.where(is_in_support, original_scale_value, 1.0)

        # 4단계: exp(Z) <= val  <=>  Z <= log(val) 이므로, 표준정규 cdf(log(val))가 원하는 확률
        probability_if_in_support = stats.norm.cdf(np.log(safe_value_for_log))

        # 5단계: support 밖(원래 값<=0)이었던 자리는 확률 0으로 덮어씀
        result = np.where(is_in_support, probability_if_in_support, 0.0)

        # 6단계: 입력 자체가 NaN이었던 자리는 결과도 NaN으로 유지 (0으로 몰래 바꾸지 않음)
        return np.where(np.isnan(u), np.nan, result)

    def ppf(self, p):
        # cdf의 역순으로 계산: 표준정규 분위수 -> exp로 원래 스케일 -> 다시 (a,s)로 표준화
        p = np.asarray(p, dtype=float)
        standard_normal_quantile = stats.norm.ppf(p)  # Phi^{-1}(p)
        original_scale_value = np.exp(standard_normal_quantile)  # exp(Z) 기준의 분위수
        return (original_scale_value - self.a) / self.s  # 표준화된 eps 기준으로 변환


ERROR_DISTRIBUTIONS = {
    "gaussian": GaussianError(),
    "student_t": StudentTError(),
    "lognormal": LogNormalError(),
}

### 검증: mean≈0, var≈1, cdf(ppf(p))≈p, LogNormal support 경계

In [4]:
rng_check = substream("validation", "error_distribution")
n_check = 2_000_000 # sample size 개수
p_grid = np.linspace(0.01, 0.99, 25) # 0.01 ~ 0.99 사이의 25개 값 생성

for name, err in ERROR_DISTRIBUTIONS.items():
    samples = err.sample(n_check, substream("validation", "error_distribution", name)) # 해당 ERROR DISTRIBUTION에서 sample 200만개 생성 
    cdf_ppf_err = np.max(np.abs(err.cdf(err.ppf(p_grid)) - p_grid))
    print(f"{name:10s} mean={samples.mean():+.4f}  var={samples.var():.4f}  max|cdf(ppf(p))-p|={cdf_ppf_err:.2e}")

ln = ERROR_DISTRIBUTIONS["lognormal"]
boundary = -ln.a / ln.s
print("\nlognormal support 경계:", "미만", ln.cdf(boundary - 0.01), " / ", "이상", ln.cdf(boundary + 0.01))

gaussian   mean=-0.0003  var=1.0000  max|cdf(ppf(p))-p|=1.11e-16
student_t  mean=+0.0008  var=0.9878  max|cdf(ppf(p))-p|=2.22e-16
lognormal  mean=-0.0011  var=0.9914  max|cdf(ppf(p))-p|=1.11e-16

lognormal support 경계: 미만 0.0  /  이상 6.290800057141992e-05


> **검증 해석 주의:** Student-$t_3$는 분산은 존재하지만 4차 적률이 존재하지 않으므로, 같은 표본 크기에서도 표본분산의 변동이 Gaussian보다 클 수 있음. 따라서 `sample.var()`가 1에 아주 가깝지 않다는 이유만으로 구현 오류라고 판단하지 않음.

## 3. DGP (Data Generating Process)

$X=(X_1,X_2)$, $X_1,X_2\overset{iid}{\sim}U(-1,1)$, $Y=m_0(X)+\sigma_0(X)\varepsilon$

- `m0_linear`, `m0_nonlinear`: true conditional mean (평균 0, 분산 4/3으로 통일)
- `sigma0_homo`, `sigma0_hetero`: true conditional scale. hetero는 $X_1$에만 의존
- `draw_X`, `draw_Y`: 설명변수와 반응변수 생성

In [5]:
def m0_linear(X):
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(2) * (x1 + x2)


def m0_nonlinear(X):
    # X1은 비선형(sin), X2는 선형, x1*x2 상호작용 포함.
    # OLS는 절편+X1+X2만 쓰므로 이 경우 OLS에 misspecification(모형 오설정)이 생김
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(6 / 11) * (2 * np.sin(np.pi * x1) + x2 + x1 * x2)


def sigma0_homo(X):
    return np.ones(X.shape[0])  # 모든 위치에서 noise 크기 동일(등분산)


def sigma0_hetero(X):
    x1 = X[:, 0]
    return (0.4 + 1.2 * np.sin(np.pi * x1) ** 2) / np.sqrt(1.18)  # X1에 따라 noise 크기가 달라짐(이분산)


MEAN_FUNCTIONS = {"linear": m0_linear, "nonlinear": m0_nonlinear}
SCALE_FUNCTIONS = {"homo": sigma0_homo, "hetero": sigma0_hetero}


def draw_X(n, rng):
    return rng.uniform(-1.0, 1.0, size=(n, 2))  # (n, 2) shape, 각 열이 U(-1,1)에서 독립


def draw_Y(X, mean_fn, scale_fn, error, rng):
    eps = error.sample(X.shape[0], rng)  # X와 독립적인 잡음
    return mean_fn(X) + scale_fn(X) * eps

### 검증: linear와 nonlinear 모두 mean 평균 0·분산 4/3, scale 최소 양수·평균제곱 1(homo, hetero)

In [6]:
rng_dgp_check = substream("validation", "dgp")
n_check = 5_000_000
X_check = draw_X(n_check, rng_dgp_check)

for name, fn in MEAN_FUNCTIONS.items():
    m = fn(X_check)
    print(f"m0_{name:10s} mean={m.mean():+.5f} (target 0)   var={m.var():.5f} (target {4/3:.4f})")

for name, fn in SCALE_FUNCTIONS.items():
    s = fn(X_check)
    target = 1.0
    print(f"sigma0_{name:8s} min={s.min():.5f} (>0)   mean(sigma^2)={np.mean(s**2):.5f} (target {target})")

m0_linear     mean=+0.00049 (target 0)   var=1.33376 (target 1.3333)
m0_nonlinear  mean=+0.00068 (target 0)   var=1.33376 (target 1.3333)
sigma0_homo     min=1.00000 (>0)   mean(sigma^2)=1.00000 (target 1.0)
sigma0_hetero   min=0.36823 (>0)   mean(sigma^2)=1.00014 (target 1.0)


### DGP 컨테이너

`mean_fn`, `scale_fn`, `error`를 하나의 시나리오로 묶음. 2 mean × 2 scale × 3 error = 12개 DGP.

In [7]:
@dataclass
class DGP:
    name: str
    mean_fn: Callable
    scale_fn: Callable
    error: object
    error_name: str  # rng_mc를 error별로 공유할 때 쓸 이름 (DGP와 불일치 방지용)
    scale_name: str  # q0(res_true 반폭)가 mean과 무관하게 scale·error에만 의존해서 캐시할 때 쓸 이름

    def sample_xy(self, n, rng):
        X = draw_X(n, rng)
        Y = draw_Y(X, self.mean_fn, self.scale_fn, self.error, rng)
        return X, Y


DGPS = {}
for mean_name, mean_fn in MEAN_FUNCTIONS.items():
    for scale_name, scale_fn in SCALE_FUNCTIONS.items():
        for error_name, error in ERROR_DISTRIBUTIONS.items():
            dgp_name = f"{mean_name}_{scale_name}_{error_name}"
            DGPS[dgp_name] = DGP(dgp_name, mean_fn, scale_fn, error, error_name, scale_name)

print(f"총 {len(DGPS)}개 DGP")
DGPS

총 12개 DGP


{'linear_homo_gaussian': DGP(name='linear_homo_gaussian', mean_fn=<function m0_linear at 0x0000022E097DDE40>, scale_fn=<function sigma0_homo at 0x0000022E097DE5C0>, error=<__main__.GaussianError object at 0x0000022E69F7F230>, error_name='gaussian', scale_name='homo'),
 'linear_homo_student_t': DGP(name='linear_homo_student_t', mean_fn=<function m0_linear at 0x0000022E097DDE40>, scale_fn=<function sigma0_homo at 0x0000022E097DE5C0>, error=<__main__.StudentTError object at 0x0000022E69F7F4D0>, error_name='student_t', scale_name='homo'),
 'linear_homo_lognormal': DGP(name='linear_homo_lognormal', mean_fn=<function m0_linear at 0x0000022E097DDE40>, scale_fn=<function sigma0_homo at 0x0000022E097DE5C0>, error=<__main__.LogNormalError object at 0x0000022E69F7F770>, error_name='lognormal', scale_name='homo'),
 'linear_hetero_gaussian': DGP(name='linear_hetero_gaussian', mean_fn=<function m0_linear at 0x0000022E097DDE40>, scale_fn=<function sigma0_hetero at 0x0000022E097DD9E0>, error=<__main__

## 4. 모델 학습 (OLS, RF)

- OLS: 절편 + $X_1,X_2$만 사용
- RF: squared-error 회귀 forest, 1000 trees. 나머지 하이퍼파라미터는 sklearn 기본값을 그대로 쓰는 pilot 기본안으로 명시

In [8]:
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

RF_HYPERPARAMS = dict(
    n_estimators=1000,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    n_jobs=1,  # 밖에서 b(training 반복)를 병렬화할 예정이라 RF 내부 병렬화는 꺼서 oversubscription 방지
)


def fit_ols(X, y):
    return LinearRegression().fit(X, y)


def fit_rf(X, y, rng):
    seed = int(rng.integers(0, 2**31 - 1))  # rng에서 정수 하나를 뽑아 sklearn의 random_state로 사용
    return RandomForestRegressor(random_state=seed, **RF_HYPERPARAMS).fit(X, y)


print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.1


### 확인: linear_homo_gaussian에서 학습해보기

$n_{train}=1000$으로 training 데이터를 만들고 OLS·RF를 학습. 데이터 생성용 rng와 RF 학습용 rng를 다른 stage로 분리해서 독립적으로 쓰는다.

In [9]:
dgp = DGPS["linear_homo_gaussian"]
b = 1

rng_train = substream(dgp.name, b, "train")
rng_model = substream(dgp.name, b, "model")

X_train, y_train = dgp.sample_xy(1000, rng_train)
ols_model = fit_ols(X_train, y_train)
rf_model = fit_rf(X_train, y_train, rng_model)

print("OLS intercept:", ols_model.intercept_, "(target 0)")
print("OLS coef     :", ols_model.coef_, f"(target both {np.sqrt(2):.4f})")

x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
print("\ntrue m0    :", dgp.mean_fn(x_eval))
print("OLS predict:", ols_model.predict(x_eval))
print("RF  predict:", rf_model.predict(x_eval))

OLS intercept: -0.015698804450583234 (target 0)
OLS coef     : [1.345043 1.456035] (target both 1.4142)

true m0    : [0.       0.       2.545584]
OLS predict: [-0.015699 -0.071195  2.505271]
RF  predict: [-0.309961 -0.518883  2.387691]


## 5. Oracle, MC 구간

- `oracle_interval`: true 오차분포의 분위수로 정확히 계산한 구간. $[m_0(x)+\sigma_0(x)q_{\alpha/2},\ m_0(x)+\sigma_0(x)q_{1-\alpha/2}]$
- `mc_interval`: 실제로 오차 표본을 $M_{MC}$개 뽑아서, 그 표본의 empirical quantile로 만든 구간 (oracle을 몬테카를로로 근사)
- `conditional_coverage`: 구간 $[L,U]$가 특정 위치 $x$에서 새 $Y$를 포함할 확률. $\{L < Y<U \}=\{L<m_0(x)+\sigma_0(x)\epsilon<U\}=\{\frac{L-m_0(x)}{\sigma_0(x)}<\epsilon<\frac{U-m_0(x)}{\sigma_0(x)}\}\Rightarrow F_\varepsilon\!\left(\frac{U-m_0(x)}{\sigma_0(x)}\right)-F_\varepsilon\!\left(\frac{L-m_0(x)}{\sigma_0(x)}\right)$ — CDF로 직접 계산하므로 새 $Y$를 뽑을 필요 없음 (CLAUDE.md §6.1, §9)

MC 표본(`eps_sample`)은 error 분포에서만 뽑고, mean/scale은 그 뒤에 위치·크기 변환으로 적용 — 그래서 같은 표본을 모든 $x$에 재사용 가능 (시뮬.md §4.3).

In [ ]:
def oracle_interval(X, dgp, alpha):
    """true 분위수를 그대로 사용해서 이론적으로 정확한 구간을 만든다.

    구간 = [m0(x) + sigma0(x)*q_lo, m0(x) + sigma0(x)*q_hi]
    여기서 q_lo, q_hi는 진짜 오차분포 eps의 양쪽 분위수.
    """
    # 1단계: 이 x에서 true mean, true scale을 계산 (우리가 DGP를 직접 설계했으니 정확히 알고 있음)
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # 2단계: 목표 alpha에 맞는 양쪽 분위수를 구함
    #        alpha=0.05면 q_lo=하위 2.5% 지점, q_hi=상위 97.5% 지점(=하위 2.5%를 뺀 지점)
    q_lo = dgp.error.ppf(alpha / 2)
    q_hi = dgp.error.ppf(1 - alpha / 2)

    # 3단계: 위치(true_mean)에 폭(true_scale * 분위수)을 더해서 구간의 양 끝을 만듦
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def draw_mc_benchmark(dgp, M_MC, rng):
    """MC 구간을 만드는 데 쓸 오차 표본을 M_MC개 뽑는다.

    이 표본은 error 분포에서만 뽑히고 mean/scale과는 무관하므로,
    같은 표본을 여러 x 위치에 재사용할 수 있다 (mc_interval에서 위치·크기 변환만 적용).
    """
    return dgp.error.sample(M_MC, rng)


def mc_interval(X, dgp, alpha, eps_sample):
    """오차 표본(eps_sample)의 empirical quantile로 oracle 구간을 몬테카를로 근사한다."""
    # 1단계: 표본에서 empirical quantile(경험적 분위수)을 계산.
    #        method="linear"는 numpy의 기본 보간 방식 -> 이 보간 규칙을 고정해서 기록해둠
    q_lo, q_hi = np.quantile(eps_sample, [alpha / 2, 1 - alpha / 2], method="linear")

    # 2단계: oracle_interval과 똑같은 방식으로, true mean/scale에 이 분위수를 적용
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def conditional_coverage(L, U, X, dgp):
    """구간 [L,U]가 특정 위치 x에서 실제로 새 Y를 포함할 확률.

    새로운 Y를 무작위로 많이 뽑아서 세는 방식이 아니라, true CDF로 정확히 계산한다
    (CLAUDE.md §6.1, §9: coverage는 CDF로 직접 계산하고 이진 포함 횟수 추정으로 대체하지 않음).
    """
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # Y = m0(X) + sigma0(X)*eps 이므로, L <= Y <= U  <=>  (L-m0)/sigma0 <= eps <= (U-m0)/sigma0
    # 즉 끝점 L, U를 "표준화된 eps 기준"으로 바꾼 뒤, 그 사이에 eps가 들어올 확률을 CDF 차이로 구함
    standardized_lower = (L - true_mean) / true_scale
    standardized_upper = (U - true_mean) / true_scale
    return dgp.error.cdf(standardized_upper) - dgp.error.cdf(standardized_lower)

### 확인: oracle vs mc, 같은 x에서 비교

In [11]:
dgp = DGPS["linear_homo_gaussian"]  # 가장 쉬운 조합(등분산 + 대칭 오차)으로 확인
alpha = 0.05
M_MC = 100_000

eps_sample = draw_mc_benchmark(dgp, M_MC, substream("mc_benchmark", dgp.error_name))

x_eval = np.array([[0.0, 0.0], [0.9, 0.9]])
L_oracle, U_oracle = oracle_interval(x_eval, dgp, alpha)
L_mc, U_mc = mc_interval(x_eval, dgp, alpha, eps_sample)

print("oracle:", np.stack([L_oracle, U_oracle], axis = 1))
print("mc    :", np.stack([L_mc, U_mc], axis=1))

print("endpoint 차이 (L,U | oracle - mc):", L_mc - L_oracle, U_mc - U_oracle)
print("oracle coverage:", conditional_coverage(L_oracle, U_oracle, x_eval, dgp), "(target 0.95)")
print("mc     coverage:", conditional_coverage(L_mc, U_mc, x_eval, dgp), "(target 0.95)")
dgp = DGPS["nonlinear_hetero_lognormal"]  # 가장 어려운 조합(이분산 + 비대칭 오차)으로 확인
alpha = 0.05
M_MC = 100_000

eps_sample = draw_mc_benchmark(dgp, M_MC, substream("mc_benchmark", dgp.error_name))

x_eval = np.array([[0.0, 0.0], [0.9, 0.9]])
L_oracle, U_oracle = oracle_interval(x_eval, dgp, alpha)
L_mc, U_mc = mc_interval(x_eval, dgp, alpha, eps_sample)

print("oracle:", np.stack([L_oracle, U_oracle], axis = 1))
print("mc    :", np.stack([L_mc, U_mc], axis=1))

print("endpoint 차이 (L,U | oracle - mc):", L_mc - L_oracle, U_mc - U_oracle)
print("oracle coverage:", conditional_coverage(L_oracle, U_oracle, x_eval, dgp), "(target 0.95)")
print("mc     coverage:", conditional_coverage(L_mc, U_mc, x_eval, dgp), "(target 0.95)")

oracle: [[-1.959964  1.959964]
 [ 0.58562   4.505548]]
mc    : [[-1.973479  1.958393]
 [ 0.572106  4.503977]]
endpoint 차이 (L,U | oracle - mc): [-0.013515 -0.013515] [-0.001571 -0.001571]
oracle coverage: [0.95 0.95] (target 0.95)
mc     coverage: [0.950687 0.950687] (target 0.95)
oracle: [[-0.256912  0.928643]
 [ 1.388856  2.914043]]
mc    : [[-0.257144  0.925289]
 [ 1.388558  2.909728]]
endpoint 차이 (L,U | oracle - mc): [-0.000232 -0.000298] [-0.003354 -0.004315]
oracle coverage: [0.95 0.95] (target 0.95)
mc     coverage: [0.950399 0.950399] (target 0.95)


### $M_{MC}$ 탐색: MC 분위수가 true 분위수에 얼마나 가까워지는가

MC의 **표준화된 분위수 오차와 coverage 오차**는 mean과 무관하고 error 분포 + $M_{MC}$ + $\alpha$로 결정됨. 다만 실제 $Y$ scale의 endpoint 오차는 $\sigma_0(x)$만큼 확대·축소됨. 따라서 raw 분위수 오차 $|\hat q-q_{true}|$와 coverage 오차를 함께 확인.

$\alpha=0.01$(양쪽 0.5%)처럼 극단적인 분위수, 그리고 Student-$t$·LogNormal처럼 꼬리가 무겁거나 비대칭인 분포가 가장 느리게 수렴할 것으로 예상 — 이 조합으로 $M_{MC}$를 늘려가며 확인.

In [12]:
M_MC_CANDIDATES = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
ALPHAS = [0.10, 0.05, 0.01]
N_REPS = 20  # 5회만으로 '안전'하다고 판단하기에는 약하므로 반복 수를 늘림

# 각 rep에서 최대 M_MC 표본을 한 번만 생성하고 작은 M_MC에서는 prefix를 재사용
# -> 원래처럼 (alpha, M_MC) 조합마다 새로 10^7개까지 생성하는 계산 낭비를 줄임
M_MC_MAX = max(M_MC_CANDIDATES)

summary = {}
for error_name in ERROR_DISTRIBUTIONS:
    for alpha in ALPHAS:
        for M_MC in M_MC_CANDIDATES:
            summary[(error_name, alpha, M_MC)] = {
                "worst_err": 0.0,
                "worst_coverage_err": 0.0,
            }

for error_name, err in ERROR_DISTRIBUTIONS.items():
    true_quantiles = {
        alpha: (err.ppf(alpha / 2), err.ppf(1 - alpha / 2))
        for alpha in ALPHAS
    }

    for rep in range(N_REPS):
        rng = substream("mc_pilot", error_name, rep)
        full_sample = err.sample(M_MC_MAX, rng)

        for M_MC in M_MC_CANDIDATES:
            sample = full_sample[:M_MC]

            for alpha in ALPHAS:
                q_lo_true, q_hi_true = true_quantiles[alpha]
                q_lo, q_hi = np.quantile(
                    sample,
                    [alpha / 2, 1 - alpha / 2],
                    method="linear",
                )

                error_lo = abs(q_lo - q_lo_true)
                error_hi = abs(q_hi - q_hi_true)
                quantile_err = max(error_lo, error_hi)

                mc_coverage = err.cdf(q_hi) - err.cdf(q_lo)
                coverage_err = abs(mc_coverage - (1 - alpha))

                key = (error_name, alpha, M_MC)
                summary[key]["worst_err"] = max(
                    summary[key]["worst_err"], quantile_err
                )
                summary[key]["worst_coverage_err"] = max(
                    summary[key]["worst_coverage_err"], coverage_err
                )

rows = []
for (error_name, alpha, M_MC), values in summary.items():
    rows.append({
        "error": error_name,
        "alpha": alpha,
        "M_MC": M_MC,
        "worst_err": values["worst_err"],
        "worst_coverage_err": values["worst_coverage_err"],
    })

mc_pilot_results = pd.DataFrame(rows)

# 기존 변수명 유지: raw quantile error 표
mc_error_table = mc_pilot_results.pivot(
    index=["error", "alpha"], columns="M_MC", values="worst_err"
)
print("[N_REPS번 중 worst raw quantile error]")
display(mc_error_table)

mc_coverage_error_table = mc_pilot_results.pivot(
    index=["error", "alpha"], columns="M_MC", values="worst_coverage_err"
)
print("\n[N_REPS번 중 worst coverage error]")
display(mc_coverage_error_table)

# heteroskedastic 조건에서 Y-scale endpoint error의 보수적 최대 배율
max_sigma_hetero = 1.6 / np.sqrt(1.18)
print(f"\nhetero max sigma = {max_sigma_hetero:.4f}")
print("Y-scale endpoint error는 raw quantile error에 sigma0(x)를 곱한 값임")

[N_REPS번 중 worst raw quantile error]


M_MC              1000      10000     100000    1000000   10000000
error     alpha                                                   
gaussian  0.0100    0.3420    0.1282    0.0314    0.0108    0.0030
          0.0500    0.2421    0.0855    0.0242    0.0055    0.0018
          0.1000    0.1611    0.0639    0.0139    0.0037    0.0017
lognormal 0.0100    1.6355    0.4497    0.2553    0.0643    0.0194
          0.0500    0.4526    0.2292    0.0513    0.0229    0.0050
          0.1000    0.4312    0.0761    0.0241    0.0111    0.0037
student_t 0.0100    1.1113    0.4227    0.1367    0.0497    0.0170
          0.0500    0.2915    0.0857    0.0384    0.0071    0.0034
          0.1000    0.2426    0.0583    0.0256    0.0046    0.0019


[N_REPS번 중 worst coverage error]


M_MC              1000      10000     100000    1000000   10000000
error     alpha                                                   
gaussian  0.0100    0.0107    0.0028    0.0006    0.0002    0.0001
          0.0500    0.0131    0.0035    0.0019    0.0005    0.0001
          0.1000    0.0170    0.0064    0.0019    0.0006    0.0002
lognormal 0.0100    0.0064    0.0014    0.0007    0.0003    0.0001
          0.0500    0.0088    0.0047    0.0017    0.0006    0.0001
          0.1000    0.0154    0.0069    0.0015    0.0007    0.0003
student_t 0.0100    0.0084    0.0026    0.0006    0.0002    0.0001
          0.0500    0.0131    0.0028    0.0015    0.0004    0.0002
          0.1000    0.0216    0.0044    0.0020    0.0004    0.0002


hetero max sigma = 1.4729
Y-scale endpoint error는 raw quantile error에 sigma0(x)를 곱한 값임


### $M_{MC}$ pilot 해석

- $\alpha$가 작을수록 더 극단적인 분위수를 추정하므로 MC quantile 오차가 크게 남을 수 있음
- Gaussian보다 Student-$t$와 LogNormal의 극단 분위수에서 수렴이 더 느림
- 최종 $M_{MC}$는 raw quantile error뿐 아니라 coverage error와 heteroskedastic 조건에서의 $Y$-scale endpoint error까지 함께 보고 결정
- 이후 structural / fitting / calibration 차이보다 MC benchmark 자체의 수치오차가 충분히 작아야 함

현재 pilot에서 확인 중인 가장 큰 후보는

$$
M_{MC}^{\mathrm{candidate}}=10{,}000{,}000
$$

이며, **최종 확정값은 아직 아님**

In [13]:
# 이후 예시에서 사용할 현재 pilot 후보값
M_MC_PILOT_CHOICE = 10_000_000

## 6. `res_true` 구간 ($q_{0,\alpha}$: population residual quantile)

True mean $m_0$은 그대로 쓰지만, 폭은 **전체 $X$ 분포에서 나온 residual** $R_0=|Y-m_0(X)|=|\sigma_0(X)\varepsilon|$의 $(1-\alpha)$ population quantile $q_{0,\alpha}$ 하나로 고정(CP 방식으로 score는 계산하는데, fitted model 대신 true model을 쓰고, finite calibration 대신 무한 calibration population을 쓴 것):

$$C_{res,0}(x) = [m_0(x)-q_{0,\alpha},\ m_0(x)+q_{0,\alpha}]$$

$q_{0,\alpha}$를 구하는 식: $H(q) = E_X\!\left[F_\varepsilon(q/\sigma_0(X)) - F_\varepsilon(-q/\sigma_0(X))\right] = 1-\alpha$

- $\sigma_0(X)$가 $X_1$에만 의존하므로(homo·hetero 둘 다), $H(q)$는 **$X_1$에 대한 1차원 수치적분**으로 높은 정밀도로 계산 가능 (`scipy.integrate.quad`)
- 그 적분값이 목표($1-\alpha$)와 같아지는 $q$를 root-finding으로 찾음 (`scipy.optimize.brentq`)
- $q_{0,\alpha}$는 **mean과 무관** — scale·error·alpha가 같으면 항상 같은 값이라 캐시 가능

In [14]:
# (scale_name, error_name, alpha) -> 이미 계산해둔 q0 값. mean과는 무관하므로 DGP 이름 전체가 아니라
# 이 세 가지로만 캐시하면 됨 (같은 scale/error/alpha면 mean이 달라도 항상 같은 q0)

# q 후보 선택 → H(q) 계산 → 1-α와 비교 → q 조정
_Q0_CACHE = {}


def _find_root_bracket(func, initial_upper=1.0, max_upper=1e6):
    """고정된 상한 50 대신, 함수값의 부호가 바뀌는 상한을 자동으로 탐색."""
    lower = 0.0
    upper = initial_upper

    while func(upper) < 0:
        upper *= 2
        if upper > max_upper:
            raise RuntimeError("root-finding 상한을 찾지 못함; DGP/분포 설정 확인 필요")

    return lower, upper

def _probability_within_q(q, x1, scale_fn, error):
    """위치 x1 하나에서, |R0| <= q 일 확률을 계산하는 가장 작은 단위 함수.

    R0 = |Y - m0(X)| = |sigma0(X) * epsilon| 이므로,
    |R0| <= q  <=>  -q/sigma0(X) <= epsilon <= q/sigma0(X)
    이 확률을 CDF 차이로 구한다.
    """
    # scale_fn은 (n,2) 모양의 X를 받아서 X[:,0](=x1)만 사용하므로, x2 자리는 0으로 채워도 결과에 영향 없음
    X_dummy = np.array([[x1, 0.0]])
    sigma0_at_x1 = scale_fn(X_dummy)[0]

    upper_side = error.cdf(q / sigma0_at_x1)
    lower_side = error.cdf(-q / sigma0_at_x1)
    return upper_side - lower_side


def _H_res_true(q, scale_fn, error):
    """H(q) = E_X1[ |R0| <= q 일 확률 ]=int_{-1}^{1} f(X_1)P(|R_0| ≤ q)dX_1. X1을 모든 위치에서 평균낸 것.

    X1 ~ U(-1,1)이므로 pdf가 항상 1/2. scipy.integrate.quad로 -1~1 구간을 적분해서
    "모든 X1에서의 확률"을 수치적분으로 높은 정밀도로 평균낸다.
    """
    def integrand(x1):
        probability_at_this_x1 = _probability_within_q(q, x1, scale_fn, error)
        return probability_at_this_x1 * 0.5  # U(-1,1)의 밀도 1/2을 곱해야 진짜 적분(평균)이 됨

    integral_value, _ = integrate.quad(integrand, -1, 1)
    return integral_value


def compute_q0(scale_name, error_name, alpha):
    """H(q) = 1-alpha 를 만족하는 q0를 찾는다 (res_true 구간의 반폭).

    H(q)는 q가 커질수록 커지는(단조증가) 함수이므로 -- q=0이면 확률 0, q가 아주 크면 확률 1에 가까움 --
    이분법 계열의 root-finding(brentq)으로 수치적으로 그 값을 찾을 수 있다.
    """
    cache_key = (scale_name, error_name, alpha)
    if cache_key in _Q0_CACHE:
        return _Q0_CACHE[cache_key]  # 이미 계산해둔 값이면 다시 계산하지 않고 재사용

    scale_fn = SCALE_FUNCTIONS[scale_name]
    error = ERROR_DISTRIBUTIONS[error_name]
    target_probability = 1 - alpha

    # H(q) - target = 0 이 되는 q를 찾는 것.
    # 상한을 50으로 고정하지 않고 실제로 부호가 바뀌는 구간을 자동 탐색.
    def h_minus_target(q):
        return _H_res_true(q, scale_fn, error) - target_probability

    lower, upper = _find_root_bracket(h_minus_target)
    q0 = optimize.brentq(h_minus_target, lower, upper, xtol=1e-10)

    _Q0_CACHE[cache_key] = q0  # 다음에 같은 (scale, error, alpha) 조합이 오면 재사용
    return q0


def res_true_interval(X, dgp, alpha):
    """[m0(x) - q0, m0(x) + q0]. true mean은 그대로 쓰고, 폭만 전체 X에서 나온 q0로 고정."""
    q0 = compute_q0(dgp.scale_name, dgp.error_name, alpha)
    true_mean = dgp.mean_fn(X)
    lower_bound = true_mean - q0
    upper_bound = true_mean + q0
    return lower_bound, upper_bound

### 검증: q0가 맞게 계산되는지 두 가지 방법으로 확인

1. **homo + 대칭분포(gaussian, student_t)에서는 이론값과 정확히 같아야 함** — $\sigma_0\equiv1$이면 $R_0=|\varepsilon|$이고, 대칭분포에서는 $|\\varepsilon|$의 $(1-\alpha)$ 분위수가 $\varepsilon$의 $(1-\alpha/2)$ 분위수와 같음. CLAUDE.md §11 "대칭·등분산에서 Oracle=res_true" 검증
2. **hetero는 독립적인 대량 Monte Carlo 표본과 비교** — 적분/root-finding 결과가 실제 표본 분위수와 일치하는지 확인

In [15]:
# 검증 1) homo에서는 q0 == error.ppf(1-alpha/2) 이어야 함 (대칭분포일 때).
#          이유: sigma0_homo(x)=1이라서 R0=|Y-m0(X)|=|eps| 그 자체이고,
#          대칭분포는 상위 (1-alpha/2) 분위수와 -하위(alpha/2) 분위수가 같으므로
#          |eps|의 (1-alpha) 분위수는 그냥 eps의 (1-alpha/2) 분위수와 같아짐.
print("[homo 검증] q0 vs true ppf(1-alpha/2)")
for error_name in ["gaussian", "student_t"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    for alpha in [0.10, 0.05, 0.01]:
        q0 = compute_q0("homo", error_name, alpha)
        q_true = error.ppf(1 - alpha / 2)
        diff = abs(q0 - q_true)
        print(f"  {error_name:10s} alpha={alpha:<5} q0={q0:.6f}  true={q_true:.6f}  diff={diff:.2e}")

# 검증 2) hetero는 이론값이 따로 없으므로, 독립적으로 뽑은 대량 MC 표본의 분위수와 비교.
#          적분+root-finding으로 구한 q0와, 실제 표본에서 직접 잰 분위수가 비슷해야 함.
print("\n[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)")

n_mc = 20_000_000
# sigma0_hetero는 X1만 사용하므로 (n,2) 배열을 만들지 않고 X1만 직접 생성
x1_check = substream("validation", "q0_hetero").uniform(-1.0, 1.0, size=n_mc)
sigma0_check = (0.4 + 1.2 * np.sin(np.pi * x1_check) ** 2) / np.sqrt(1.18)
del x1_check

for error_name in ["gaussian", "student_t", "lognormal"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    # 별도 R0 배열을 추가로 만들지 않고 eps_check 배열을 residual 저장용으로 재사용
    eps_check = error.sample(n_mc, substream("validation", "q0_hetero", error_name))
    eps_check *= sigma0_check
    np.abs(eps_check, out=eps_check)  # 이제 eps_check가 R0_check 역할

    for alpha in [0.10, 0.05, 0.01]:
        q0_from_integral = compute_q0("hetero", error_name, alpha)
        q0_from_mc_sample = np.quantile(eps_check, 1 - alpha)
        diff = abs(q0_from_integral - q0_from_mc_sample)
        print(
            f"  {error_name:10s} alpha={alpha:<5} "
            f"q0(적분)={q0_from_integral:.6f}  q0(MC)={q0_from_mc_sample:.6f}  diff={diff:.2e}"
        )

del sigma0_check, eps_check

[homo 검증] q0 vs true ppf(1-alpha/2)
  gaussian   alpha=0.1   q0=1.644854  true=1.644854  diff=2.44e-15
  gaussian   alpha=0.05  q0=1.959964  true=1.959964  diff=1.55e-15
  gaussian   alpha=0.01  q0=2.575829  true=2.575829  diff=4.54e-12
  student_t  alpha=0.1   q0=1.358715  true=1.358715  diff=2.10e-11
  student_t  alpha=0.05  q0=1.837386  true=1.837386  diff=4.60e-14
  student_t  alpha=0.01  q0=3.372251  true=3.372251  diff=1.21e-11

[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)
  gaussian   alpha=0.1   q0(적분)=1.673710  q0(MC)=1.674199  diff=4.89e-04
  gaussian   alpha=0.05  q0(적분)=2.127187  q0(MC)=2.127300  diff=1.13e-04
  gaussian   alpha=0.01  q0(적분)=3.051525  q0(MC)=3.050796  diff=7.29e-04
  student_t  alpha=0.1   q0(적분)=1.328576  q0(MC)=1.328330  diff=2.46e-04
  student_t  alpha=0.05  q0(적분)=1.857449  q0(MC)=1.856874  diff=5.75e-04
  student_t  alpha=0.01  q0(적분)=3.527012  q0(MC)=3.528164  diff=1.15e-03
  lognormal  alpha=0.1   q0(적분)=0.939333  q0(MC)=0.939555  diff=2.22e-04
  lo

**$q_0$ 검증 해석**

- homo + 대칭분포에서는 적분/root-finding 결과가 알려진 이론값과 사실상 동일해야 함
- hetero에서는 닫힌형태의 단순한 기준값이 없으므로, 독립적인 대량 MC residual 표본의 empirical quantile과 비교
- 여기서 $n_{mc}=20{,}000{,}000$은 **$C_{res,0}$ 본 계산에 쓰는 값이 아니라 `compute_q0()` 구현을 검증하기 위한 일회성 표본 크기**
- 본 시뮬레이션의 $C_{res,0}$ 계산에는 계속 수치적분 + root-finding으로 얻은 $q_0$ 사용

## 7. `pop` 구간 ($q_{pop,j,\alpha}$: fitted model 기준 population residual quantile)

Training으로 학습한 fitted model $\widehat m_j$를 고정하고, **fitted model 기준 residual** $R_j=|Y-\widehat m_j(X)|$의 population quantile로 폭을 정함:

$$C_{pop,j}(x) = [\widehat m_j(x)-q_{pop,j,\alpha},\ \widehat m_j(x)+q_{pop,j,\alpha}]$$

`res_true`(§6)와 식은 똑같지만($g=m_0$ 대신 $g=\widehat m_j$), 계산 방법이 다름:
- `res_true`는 $\sigma_0(X)$가 $X_1$에만 의존해서 1차원 **고정밀 수치적분**으로 계산 가능
- `pop`은 $\widehat m_j(X)$가 $X_1,X_2$ 둘 다에 의존하므로, 적분으로 직접 계산하기가 훨씬 까다로움 -> $$ X_1^*,\ldots,X_{M_{\text{pop}}}^* \sim P_X $$를 많이 뽑아서 $X$에 대한 평균을 Monte Carlo로 근사->
**독립적인 reference 입력 표본**(`M_pop`개)으로 $H_g(q)=E_X[p_g(q,X)]$를 몬테카를로 근사 (CLAUDE.md §5)

Reference 입력은 training·calibration·평가 입력과 독립이고, **같은 fitted model이면 OLS·RF가 이 reference를 공유** — 모델 예측값만 다르고 reference 위치는 같음.

In [16]:
def draw_population_reference(M_pop, rng):
    """pop 구간용 독립 reference 입력. training·calibration·평가 입력과 무관하며,
    같은 (dgp, b)에서 OLS·RF가 이 표본을 공유한다."""
    return draw_X(M_pop, rng)


def _H_pop(q, cached_g, cached_true_mean, cached_true_scale, error):
    """H_g(q) = E_X[p_g(q,X)]를, 미리 계산해둔(cached) reference 값들로 근사.

    cached_g, cached_true_mean, cached_true_scale은 reference 입력에서 한 번만 계산해두고
    q를 바꿔가며(root-finding) 여러 번 재사용 -> fitted_model.predict를 반복 호출하지 않음
    (CLAUDE.md §9: 예측은 동일 입력에서 캐시하고 재사용).
    """
    upper_z = (cached_g + q - cached_true_mean) / cached_true_scale
    lower_z = (cached_g - q - cached_true_mean) / cached_true_scale
    p_values = error.cdf(upper_z) - error.cdf(lower_z)  # reference 각 점에서의 "포함 확률"
    return p_values.mean()  # 전체 reference에 대한 평균 = H_g(q)의 근사값


def compute_q_pop(fitted_model, ref_X, dgp, alpha):
    """H_g(q) = 1-alpha 를 만족하는 q_pop을 찾는다 (pop 구간의 반폭)."""
    # 1단계: reference 입력에서 필요한 값들을 딱 한 번만 계산해서 캐시
    g_at_ref = fitted_model.predict(ref_X)  # g(x) = fitted model의 예측값
    true_mean_at_ref = dgp.mean_fn(ref_X)
    true_scale_at_ref = dgp.scale_fn(ref_X)

    # 2단계: 그 캐시를 이용해 H(q)-target = 0 이 되는 q를 root-finding으로 찾음 (res_true와 같은 방식)
    target_probability = 1 - alpha

    def h_minus_target(q):
        return _H_pop(q, g_at_ref, true_mean_at_ref, true_scale_at_ref, dgp.error) - target_probability

    lower, upper = _find_root_bracket(h_minus_target)
    return optimize.brentq(h_minus_target, lower, upper, xtol=1e-10)


def pop_interval(X, fitted_model, q_pop):
    """[g(x) - q_pop, g(x) + q_pop]. fitted model의 예측값을 중심으로, 폭은 q_pop으로 고정."""
    g = fitted_model.predict(X)
    return g - q_pop, g + q_pop

### 확인: q_pop vs q0, 그리고 M_pop 안정성

In [ ]:
print("\n[M_pop 안정성 확인: 조건, alpha=0.01]")
# mean, sd가 안정되는지 확인

# M_pop 후보
M_POP_CANDIDATES = [20_000, 50_000, 100_000, 200_000, 500_000, 1_000_000]

# 가장 큰 reference sample을 한 번 만든 뒤
# 작은 M_pop은 앞부분(prefix)만 잘라서 사용
M_POP_MAX = max(M_POP_CANDIDATES)

# reference sample을 바꿔가며 안정성 확인
N_REPS_POP = 10

for check_dgp_name in ["linear_homo_gaussian", "nonlinear_hetero_student_t", "nonlinear_hetero_lognormal"]:
    check_dgp = DGPS[check_dgp_name]
    check_b = 1

    # ------------------------------------------------------------
    # 1. 이 DGP에서 training data 생성 및 OLS / RF 학습
    #    training은 모든 M_pop 후보에서 동일하게 고정
    # ------------------------------------------------------------
    check_X_train, check_y_train = check_dgp.sample_xy(1000, substream(check_dgp.name, check_b, "train"))

    check_ols = fit_ols(check_X_train, check_y_train)
    check_rf = fit_rf(check_X_train, check_y_train, substream(check_dgp.name, check_b, "model"))

    # ------------------------------------------------------------
    # 2. M_pop별 q_pop 결과를 저장할 공간
    # ------------------------------------------------------------
    results = {
        M_pop: {
            "OLS": [],
            "RF": [],
        } for M_pop in M_POP_CANDIDATES
    }

    # ------------------------------------------------------------
    # 3. reference sample을 N_REPS_POP번 독립적으로 생성
    #
    # 중요한 점:
    # 같은 rep 안에서는 M_POP_MAX=1,000,000개를 한 번만 생성하고
    # 각 M_pop에서는 그 앞부분만 사용
    #
    # 따라서
    # 20,000 ⊂ 50,000 ⊂ ... ⊂ 1,000,000
    # 형태로 같은 reference sample을 공유
    # ------------------------------------------------------------
    for rep in range(N_REPS_POP):
        check_ref_X_full = draw_population_reference(M_POP_MAX, substream(check_dgp.name, check_b, "reference", "M_pop_check", rep))   # M_pop은 seed에 넣지 않음

        # --------------------------------------------------------
        # 4. 같은 reference sample의 prefix를 사용해
        #    각 M_pop에서 q_pop 계산
        # --------------------------------------------------------
        for M_pop in M_POP_CANDIDATES:
            check_ref_X = check_ref_X_full[:M_pop]

            q_pop_ols = compute_q_pop(check_ols, check_ref_X, check_dgp, 0.01)
            q_pop_rf = compute_q_pop(check_rf, check_ref_X, check_dgp, 0.01)

            results[M_pop]["OLS"].append(q_pop_ols)
            results[M_pop]["RF"].append(q_pop_rf)

    # ------------------------------------------------------------
    # 5. 결과 출력
    # ------------------------------------------------------------
    print(f"\n  {check_dgp_name}")

    for M_pop in M_POP_CANDIDATES:
        q_ols_reps = np.array(results[M_pop]["OLS"])
        q_rf_reps = np.array(results[M_pop]["RF"])

        print(
            f"    M_pop={M_pop:>9,} | "
            f"OLS mean={np.mean(q_ols_reps):.5f}, "
            f"sd={np.std(q_ols_reps, ddof=1):.2e} | "   
            f"RF mean={np.mean(q_rf_reps):.5f}, "
            f"sd={np.std(q_rf_reps, ddof=1):.2e}"
        )


[추가 M_pop 안정성 확인: 어려운 조건, alpha=0.01]

  linear_homo_gaussian
    M_pop=   20,000 | OLS mean=2.57894, sd=2.13e-05 | RF mean=2.81510, sd=3.23e-03
    M_pop=   50,000 | OLS mean=2.57895, sd=1.08e-05 | RF mean=2.81408, sd=1.69e-03
    M_pop=  100,000 | OLS mean=2.57895, sd=7.71e-06 | RF mean=2.81406, sd=8.61e-04
    M_pop=  200,000 | OLS mean=2.57895, sd=6.52e-06 | RF mean=2.81478, sd=7.05e-04
    M_pop=  500,000 | OLS mean=2.57895, sd=6.77e-06 | RF mean=2.81474, sd=4.06e-04
    M_pop=1,000,000 | OLS mean=2.57895, sd=3.86e-06 | RF mean=2.81484, sd=2.77e-04

  nonlinear_hetero_student_t
    M_pop=   20,000 | OLS mean=3.77036, sd=1.61e-02 | RF mean=3.66091, sd=1.67e-02
    M_pop=   50,000 | OLS mean=3.77038, sd=8.56e-03 | RF mean=3.66265, sd=7.64e-03
    M_pop=  100,000 | OLS mean=3.77058, sd=5.37e-03 | RF mean=3.66422, sd=6.39e-03
    M_pop=  200,000 | OLS mean=3.76969, sd=2.72e-03 | RF mean=3.66259, sd=3.69e-03
    M_pop=  500,000 | OLS mean=3.76979, sd=2.00e-03 | RF mean=3.66268, sd=2.0

$C_{pop,j}$의 true $q_{pop,j}$는 일반적으로 닫힌형태로 직접 계산하기 어려움  
→ 후보 $M_{pop}$보다 훨씬 큰 독립 reference sample $M_{ref}=10,000,000$으로 high-precision benchmark $q_{pop,j}^{ref}$ 계산  
→ 각 후보에 대해 $|\widehat q_{pop,j}(M_{pop})-q_{pop,j}^{ref}|$ 확인

In [18]:
# ============================================================
# M_pop 선택 pilot
# - 큰 reference sample로 q_pop^ref를 만든 뒤
# - 각 M_pop에서 계산한 q_pop이 q_pop^ref에 얼마나 가까워지는지 확인
# ============================================================

print("\n[M_pop 정확도 확인: high-precision reference와 비교, alpha=0.01]")

alpha = 0.01

# 실제 후보
M_POP_CANDIDATES = [20_000, 50_000, 100_000, 200_000, 500_000, 1_000_000]
M_POP_MAX = max(M_POP_CANDIDATES)

# 후보 M_pop의 Monte Carlo 변동 확인용 반복
N_REPS_POP = 10

# ------------------------------------------------------------
# q_pop의 '진짜 값'을 직접 알 수 없으므로
# 후보 M_pop보다 훨씬 큰 independent reference sample을
# high-precision benchmark로 사용
#
# 주의:
# q_pop_ref는 mathematical exact truth가 아니라
# 매우 정밀한 Monte Carlo reference
# ------------------------------------------------------------
M_REF = 10_000_000

for check_dgp_name in ["linear_homo_gaussian", "nonlinear_hetero_student_t", "nonlinear_hetero_lognormal"]:
    check_dgp = DGPS[check_dgp_name]
    check_b = 1

    print(f"\n{'=' * 80}")
    print(f"DGP: {check_dgp_name}")
    print(f"{'=' * 80}")

    # ========================================================
    # 1. Training data 한 번 생성
    #    모든 M_pop과 reference benchmark에서 동일한 fitted model 사용
    # ========================================================
    check_X_train, check_y_train = check_dgp.sample_xy(1000, substream(check_dgp.name, check_b, "train"))

    check_ols = fit_ols(check_X_train, check_y_train)
    check_rf = fit_rf(check_X_train, check_y_train, substream(check_dgp.name, check_b, "model"))

    # ========================================================
    # 2. High-precision reference q_pop 계산
    #
    # 후보 M_pop에서 사용하는 reference와 완전히 독립적인 seed 사용
    # ========================================================
    print(f"\n[High-precision reference 계산: M_ref={M_REF:,}]")

    ref_X_truth = draw_population_reference(M_REF, substream(check_dgp.name, check_b, "reference", "high_precision_truth"))

    q_ref_ols = compute_q_pop(check_ols, ref_X_truth, check_dgp, alpha)
    q_ref_rf = compute_q_pop(check_rf, ref_X_truth, check_dgp, alpha)

    print(f"  OLS q_pop_ref = {q_ref_ols:.6f}")
    print(f"  RF  q_pop_ref = {q_ref_rf:.6f}")

    # 큰 reference sample은 더 이상 필요 없음
    del ref_X_truth

    # ========================================================
    # 3. 각 M_pop 결과 저장
    # ========================================================
    results = {
        M_pop: {
            "OLS": [],
            "RF": [],
        } for M_pop in M_POP_CANDIDATES
    }

    # ========================================================
    # 4. reference sample을 10번 독립적으로 생성
    #
    # 같은 rep 안에서는 prefix 사용
    #
    # 20k ⊂ 50k ⊂ 100k ⊂ ... ⊂ 1M
    # ========================================================
    for rep in range(N_REPS_POP):

        check_ref_X_full = draw_population_reference(M_POP_MAX, substream(check_dgp.name, check_b, "reference", "M_pop_check", rep))

        for M_pop in M_POP_CANDIDATES:
            check_ref_X = check_ref_X_full[:M_pop]

            q_pop_ols = compute_q_pop(check_ols, check_ref_X, check_dgp, alpha)
            q_pop_rf = compute_q_pop(check_rf, check_ref_X, check_dgp, alpha)

            results[M_pop]["OLS"].append(q_pop_ols)
            results[M_pop]["RF"].append(q_pop_rf)

        del check_ref_X_full

    # ========================================================
    # 5. q_ref와 직접 비교
    # ========================================================
    print("\n[결과]")
    print(
        f"{'M_pop':>10} | "
        f"{'OLS mean':>10} {'sd':>10} {'MAE(ref)':>10} {'MAX(ref)':>10} | "
        f"{'RF mean':>10} {'sd':>10} {'MAE(ref)':>10} {'MAX(ref)':>10}"
    )

    for M_pop in M_POP_CANDIDATES:
        q_ols = np.asarray(results[M_pop]["OLS"])
        q_rf = np.asarray(results[M_pop]["RF"])

        # ----------------------------------------------------
        # reference benchmark와의 absolute error
        # ----------------------------------------------------
        err_ols = np.abs(q_ols - q_ref_ols)
        err_rf = np.abs(q_rf - q_ref_rf)

        print(
            f"{M_pop:>10,} | "
            f"{np.mean(q_ols):>10.5f} "
            f"{np.std(q_ols, ddof=1):>10.2e} "
            f"{np.mean(err_ols):>10.2e} "
            f"{np.max(err_ols):>10.2e} | "
            f"{np.mean(q_rf):>10.5f} "
            f"{np.std(q_rf, ddof=1):>10.2e} "
            f"{np.mean(err_rf):>10.2e} "
            f"{np.max(err_rf):>10.2e}"
        )


[M_pop 정확도 확인: high-precision reference와 비교, alpha=0.01]

DGP: linear_homo_gaussian

[High-precision reference 계산: M_ref=10,000,000]
  OLS q_pop_ref = 2.578949
  RF  q_pop_ref = 2.814838

[결과]
     M_pop |   OLS mean         sd   MAE(ref)   MAX(ref) |    RF mean         sd   MAE(ref)   MAX(ref)
    20,000 |    2.57894   2.13e-05   1.92e-05   4.27e-05 |    2.81510   3.23e-03   2.45e-03   5.31e-03
    50,000 |    2.57895   1.08e-05   9.27e-06   1.55e-05 |    2.81408   1.69e-03   1.45e-03   3.73e-03
   100,000 |    2.57895   7.71e-06   5.99e-06   1.45e-05 |    2.81406   8.61e-04   9.62e-04   1.89e-03
   200,000 |    2.57895   6.52e-06   5.24e-06   1.13e-05 |    2.81478   7.05e-04   5.51e-04   1.36e-03
   500,000 |    2.57895   6.77e-06   5.23e-06   1.23e-05 |    2.81474   4.06e-04   3.34e-04   6.59e-04
 1,000,000 |    2.57895   3.86e-06   3.19e-06   6.46e-06 |    2.81484   2.77e-04   2.06e-04   4.34e-04

DGP: nonlinear_hetero_student_t

[High-precision reference 계산: M_ref=10,000,000]
  O

이론적으로 $\infty$ population expectation을 충분히 잘 근사할 reference sample 크기 $M_{pop}$을 찾는 것

### 이후 구간 예시 계산용 설정

최종 $M_{pop}$은 pilot 결과를 확인한 뒤 확정 예정

In [19]:
M_POP_FINAL = 1_000_000  # 최종값 확정 후 정수로 입력
# ------------------------------------------------------------
# 중요:
# 바로 앞 high-precision pilot에서 alpha=0.01을 사용했으므로
# 이후 예시는 다시 linear-homo-Gaussian, alpha=0.05로 명시적으로 reset
# ------------------------------------------------------------
dgp = DGPS["linear_homo_gaussian"]
alpha = 0.05
b = 1

rng_train = substream(dgp.name, b, "train")
rng_model = substream(dgp.name, b, "model")

X_train, y_train = dgp.sample_xy(1000, rng_train)
ols_model = fit_ols(X_train, y_train)
rf_model = fit_rf(X_train, y_train, rng_model)

q0 = compute_q0(dgp.scale_name, dgp.error_name, alpha)

# pop 예시용 독립 reference
rng_reference = substream(dgp.name, b, "reference", "example")
ref_X = draw_population_reference(M_POP_FINAL, rng_reference)

q_pop_ols = compute_q_pop(ols_model, ref_X, dgp, alpha)
q_pop_rf = compute_q_pop(rf_model, ref_X, dgp, alpha)

print(f"\n[예시용 M_pop={M_POP_FINAL:,}, 최종값 아님]")
print(f"q0 (res_true) = {q0:.5f}")
print(f"q_pop(OLS) = {q_pop_ols:.5f}  (q_pop - q0 = {q_pop_ols - q0:+.5f}, >= 0 이어야 함)")
print(f"q_pop(RF)  = {q_pop_rf:.5f}  (q_pop - q0 = {q_pop_rf - q0:+.5f}, >= 0 이어야 함)")

# ---- 여기까지는 "폭"(q_pop)만 구한 것. 실제 구간 C_pop,j(x) = [g(x)-q_pop, g(x)+q_pop] ----
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])

L_pop_ols, U_pop_ols = pop_interval(x_eval, ols_model, q_pop_ols)
L_pop_rf, U_pop_rf = pop_interval(x_eval, rf_model, q_pop_rf)

print("\n실제 C_pop,j(x) 구간 (x_eval의 각 행마다 [L, U]):")
print("pop(OLS):", np.stack([L_pop_ols, U_pop_ols], axis=1))
print("pop(RF) :", np.stack([L_pop_rf, U_pop_rf], axis=1))

# 참고: res_true 구간과 중심 비교
L_res, U_res = res_true_interval(x_eval, dgp, alpha)
print("res_true:", np.stack([L_res, U_res], axis=1))


[예시용 M_pop=1,000,000, 최종값 아님]
q0 (res_true) = 1.95996
q_pop(OLS) = 1.96234  (q_pop - q0 = +0.00238, >= 0 이어야 함)
q_pop(RF)  = 2.14041  (q_pop - q0 = +0.18045, >= 0 이어야 함)

실제 C_pop,j(x) 구간 (x_eval의 각 행마다 [L, U]):
pop(OLS): [[-1.978038  1.946641]
 [-2.033535  1.891144]
 [ 0.542932  4.467611]]
pop(RF) : [[-2.450373  1.830451]
 [-2.659295  1.621529]
 [ 0.247279  4.528103]]
res_true: [[-1.959964  1.959964]
 [-1.959964  1.959964]
 [ 0.58562   4.505548]]


## 8. `cp` 구간 (실제 finite-sample Split Conformal Prediction)

앞의 `pop`은 "calibration이 무한히 많다"는 가정이었는데, `cp`는 **실제 유한한 calibration 데이터 $m$개**로 만드는 진짜 conformal 구간입니다:

$$C_{CP,j}(x) = [\widehat m_j(x)-\widehat q_{j,\alpha},\ \widehat m_j(x)+\widehat q_{j,\alpha}]$$

$\widehat q_{j,\alpha}$를 구하는 방법은 root-finding이 아니라 훨씬 단순합니다:

1. calibration $m$개의 절대잔차 $R_i=|Y_i-\widehat m_j(X_i)|$를 오름차순 정렬
2. 순위 $k_\alpha=\lceil (m+1)(1-\alpha)\rceil$ 계산
3. $k_\alpha\le m$이면 정렬한 배열의 $k_\alpha$번째(1-indexed) 값 사용, $k_\alpha=m+1$이면 $+\infty$

$m=1000$이면 순위는 alpha=[0.10,0.05,0.01]에 대해 정확히 **901, 951, 991**이어야 합니다 (CLAUDE.md §4, §11 필수 검증 항목). numpy 기본 quantile 보간은 이 규칙과 다르므로 절대 쓰지 않고, 정렬된 배열에서 정확한 순서통계량을 직접 뽑습니다.

In [20]:
def cp_quantile(fitted_model, X_cal, y_cal, alpha):
    """실제 calibration 데이터로 conformal quantile(반폭) q_hat을 계산.

    반환값: (q_hat, info). info에는 순위 k, calibration 크기 m, 상태(finite/infinite)를 담아서
    k=m+1(무한 구간)인지 나중에 명확히 구분할 수 있게 함.
    """
    # 1단계: 절대잔차 계산 (같은 fitted model이라도 pop과 달리 실제 데이터 m개만 사용)
    residuals = np.abs(y_cal - fitted_model.predict(X_cal))
    sorted_residuals = np.sort(residuals)  # 오름차순 정렬
    m = len(sorted_residuals)

    # 2단계: conformal 순위 계산
    # 현재 설정(alpha=0.10, 0.05, 0.01; m=1000)에서는 임의의 -1e-9 보정이 필요하지 않음
    k = math.ceil((m + 1) * (1 - alpha))

    # 3단계: k가 calibration 크기를 넘으면(=m+1) 무한 구간, 아니면 k번째(1-indexed) 값 사용
    if k > m:
        q_hat = np.inf
        state = "infinite"
    else:
        q_hat = sorted_residuals[k - 1]  # 1-indexed k번째 = 0-indexed (k-1)번째
        state = "finite"

    info = {"k": k, "m": m, "state": state}
    return q_hat, info


def cp_interval(X, fitted_model, q_hat):
    """[g(x) - q_hat, g(x) + q_hat]. pop_interval과 형태는 같고 폭(q_hat)만 다른 방식으로 구한 것."""
    g = fitted_model.predict(X)
    return g - q_hat, g + q_hat

### 검증: 순위 901/951/991, 그리고 $k=m+1$ 경계

CLAUDE.md §11 필수 검증 항목. 알려진 배열(1,2,...,m)로 순위가 정확한지 확인하고, calibration이 작고 alpha가 작아서 $k$가 $m$을 넘는 경우(무한 구간)도 확인.

In [21]:
class _ZeroModel:
    """검증용 가짜 모델: 항상 0을 예측 -> residual = |y - 0| = y 그대로 되어 known array를 검증에 쓸 수 있음."""

    def predict(self, X):
        return np.zeros(len(X))


_zero_model = _ZeroModel()

# 1) m=1000, 알려진 배열 [1,2,...,1000]으로 순위가 정확히 901/951/991인지 확인
known_residuals = np.arange(1, 1001, dtype=float)
X_dummy = np.zeros((1000, 2))
print("[m=1000 순위 검증]")
for alpha, expected_k in zip([0.10, 0.05, 0.01], [901, 951, 991]):
    q_hat, info = cp_quantile(_zero_model, X_dummy, known_residuals, alpha)
    print(f"  alpha={alpha:<5} k={info['k']} (target {expected_k})  q_hat={q_hat} (target {expected_k})")
    assert info["k"] == expected_k, f"순위 불일치: {info['k']} != {expected_k}"
    assert q_hat == expected_k, f"q_hat 불일치: {q_hat} != {expected_k}"

# 2) k = m+1 경계: calibration이 작고 alpha가 아주 작으면 무한 구간이 나와야 함
print("\n[k=m+1 경계 검증]")
small_residuals = np.arange(1, 11, dtype=float)  # m=10
X_dummy_small = np.zeros((10, 2))
q_hat, info = cp_quantile(_zero_model, X_dummy_small, small_residuals, alpha=0.01)
print(f"  m=10, alpha=0.01: k={info['k']}, m={info['m']}, q_hat={q_hat}, state={info['state']}")
assert info["state"] == "infinite" and np.isinf(q_hat)

print("\nCP 순위·경계 검증 통과")

[m=1000 순위 검증]
  alpha=0.1   k=901 (target 901)  q_hat=901.0 (target 901)
  alpha=0.05  k=951 (target 951)  q_hat=951.0 (target 951)
  alpha=0.01  k=991 (target 991)  q_hat=991.0 (target 991)

[k=m+1 경계 검증]
  m=10, alpha=0.01: k=11, m=10, q_hat=inf, state=infinite

CP 순위·경계 검증 통과


### 확인: 실제 calibration으로 cp 구간 만들기, pop과 비교

같은 `linear_homo_gaussian`의 `ols_model`/`rf_model`(b=1)에, 실제 독립 calibration $m=1000$개를 만들어 cp 구간을 계산. `cp`는 유한 calibration을 쓰므로 `pop`(calibration 무한 가정)보다 표본 변동이 있어야 하지만, $m=1000$이면 비슷한 값이 나올 것으로 예상.

In [22]:
dgp = DGPS["linear_homo_gaussian"]  # ols_model, rf_model은 §4에서 이 dgp의 b=1로 학습해둔 것
b, r = 1, 1
alpha = 0.05

# 실제 독립 calibration 데이터 (같은 (b, r)에서 OLS, RF가 이 calibration을 공유)
rng_cal = substream(dgp.name, b, r, "calibration")
X_cal, y_cal = dgp.sample_xy(1000, rng_cal)

q_hat_ols, info_ols = cp_quantile(ols_model, X_cal, y_cal, alpha)
q_hat_rf, info_rf = cp_quantile(rf_model, X_cal, y_cal, alpha)
print(f"cp: q_hat(OLS)={q_hat_ols:.5f}  {info_ols}")
print(f"cp: q_hat(RF) ={q_hat_rf:.5f}  {info_rf}")

# 앞서(§7) 계산해둔 q_pop과 비교 -- 값이 유한 calibration 표본변동만큼만 다르면 정상
print(f"\npop: q_pop(OLS)={q_pop_ols:.5f}   cp - pop = {q_hat_ols - q_pop_ols:+.5f}")
print(f"pop: q_pop(RF) ={q_pop_rf:.5f}   cp - pop = {q_hat_rf - q_pop_rf:+.5f}")

# 실제 구간까지 만들어서 확인
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
L_cp_ols, U_cp_ols = cp_interval(x_eval, ols_model, q_hat_ols)
L_cp_rf, U_cp_rf = cp_interval(x_eval, rf_model, q_hat_rf)
print("\ncp(OLS):", np.stack([L_cp_ols, U_cp_ols], axis=1))
print("cp(RF) :", np.stack([L_cp_rf, U_cp_rf], axis=1))

cp: q_hat(OLS)=1.97794  {'k': 951, 'm': 1000, 'state': 'finite'}
cp: q_hat(RF) =2.12029  {'k': 951, 'm': 1000, 'state': 'finite'}

pop: q_pop(OLS)=1.96234   cp - pop = +0.01560
pop: q_pop(RF) =2.14041   cp - pop = -0.02012

cp(OLS): [[-1.993635  1.962237]
 [-2.049131  1.906741]
 [ 0.527335  4.483207]]
cp(RF) : [[-2.430256  1.810334]
 [-2.639178  1.601412]
 [ 0.267396  4.507986]]


## 9. 다섯 구간 한눈에 비교하는 표

`linear_homo_gaussian`, $\alpha=0.05$, `x_eval`의 세 위치에서 다섯 구간(oracle·mc·res_true·pop·cp)을 전부 계산해서 하나의 표로 정리. `oracle`/`mc`/`res_true`는 모델과 무관하므로 `model_id="-"`, `pop`/`cp`는 OLS·RF 각각.

In [23]:
dgp = DGPS["linear_homo_gaussian"]  # ols_model, rf_model, q0, q_pop_*, q_hat_*는 위에서 이미 계산해둔 값들
alpha = 0.05
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])

# mc만 이 dgp(error=gaussian)에 맞춰 새로 뽑음
# 현재는 최종 확정 전이므로 M_MC_PILOT_CHOICE를 사용
eps_sample_gaussian = draw_mc_benchmark(
    dgp,
    M_MC_PILOT_CHOICE,
    substream("mc_benchmark", dgp.error_name, "five_interval_table"),
)

# (interval_id, model_id, (L, U)) 튜플을 하나씩 나열 -- model_id가 없는(None) 구간은 표에서 "-"로 표시
families = [
    ("oracle", None, oracle_interval(x_eval, dgp, alpha)),
    ("mc", None, mc_interval(x_eval, dgp, alpha, eps_sample_gaussian)),
    ("res_true", None, res_true_interval(x_eval, dgp, alpha)),
    ("pop", "OLS", pop_interval(x_eval, ols_model, q_pop_ols)),
    ("pop", "RF", pop_interval(x_eval, rf_model, q_pop_rf)),
    ("cp", "OLS", cp_interval(x_eval, ols_model, q_hat_ols)),
    ("cp", "RF", cp_interval(x_eval, rf_model, q_hat_rf)),
]

rows = []
for interval_id, model_id, (L, U) in families:
    coverage = conditional_coverage(L, U, x_eval, dgp)
    for i in range(len(x_eval)):
        rows.append({
            "interval_id": interval_id,
            "model_id": model_id or "-",
            "x1": x_eval[i, 0],
            "x2": x_eval[i, 1],
            "L": L[i],
            "U": U[i],
            "center": (L[i] + U[i]) / 2,
            "half_width": (U[i] - L[i]) / 2,
            "coverage": coverage[i],
        })

five_intervals_table = pd.DataFrame(rows)
five_intervals_table

,interval_id,model_id,x1,x2,L,U,center,half_width,coverage
0,oracle,-,0.0000,0.0000,-1.9600,1.9600,-0.0000,1.9600,0.9500
1,oracle,-,0.5000,-0.5000,-1.9600,1.9600,-0.0000,1.9600,0.9500
2,oracle,-,0.9000,0.9000,0.5856,4.5055,2.5456,1.9600,0.9500
3,mc,-,0.0000,0.0000,-1.9605,1.9598,-0.0004,1.9602,0.9500
4,mc,-,0.5000,-0.5000,-1.9605,1.9598,-0.0004,1.9602,0.9500
5,mc,-,0.9000,0.9000,0.5851,4.5054,2.5452,1.9602,0.9500
6,res_true,-,0.0000,0.0000,-1.9600,1.9600,0.0000,1.9600,0.9500
7,res_true,-,0.5000,-0.5000,-1.9600,1.9600,0.0000,1.9600,0.9500
8,res_true,-,0.9000,0.9000,0.5856,4.5055,2.5456,1.9600,0.9500
9,pop,OLS,0.0000,0.0000,-1.9780,1.9466,-0.0157,1.9623,0.9502


### 같은 표를 pivot으로 정리 (지표별로 한눈에 비교)

`interval_id`+`model_id`를 합쳐 하나의 열 이름(`family`, 예: `pop_RF`)으로 만들고, 위치(x1,x2)를 행으로 두고 각 지표(center, half_width, coverage)마다 별도의 작은 표를 만든다. 열 순서는 oracle→mc→res_true→pop_OLS→pop_RF→cp_OLS→cp_RF로 고정해서 원래 5개 구간 순서를 그대로 따라가게 함.

In [24]:
FAMILY_ORDER = ["oracle", "mc", "res_true", "pop_OLS", "pop_RF", "cp_OLS", "cp_RF"]

# interval_id와 model_id를 합쳐서 "family" 하나의 이름으로 만듦 (model_id가 "-"면 그냥 interval_id만 씀)
table_with_family = five_intervals_table.copy()
table_with_family["family"] = table_with_family.apply(
    lambda row: row["interval_id"] if row["model_id"] == "-" else f"{row['interval_id']}_{row['model_id']}",
    axis=1,
)

# 지표(center, half_width, coverage)마다 따로 pivot 표를 만들고, 각각을 별도의 표로 보여줌
center_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="center")[FAMILY_ORDER]
half_width_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="half_width")[FAMILY_ORDER]
coverage_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="coverage")[FAMILY_ORDER]

print("=== center ===")
display(center_table)

print("\n=== half_width ===")
display(half_width_table)

print("\n=== coverage (target 0.95) ===")
display(coverage_table)

=== center ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,-0.0000,-0.0004,0.0000,-0.0157,-0.3100,-0.0157,-0.3100
0.5000,-0.5000,-0.0000,-0.0004,0.0000,-0.0712,-0.5189,-0.0712,-0.5189
0.9000,0.9000,2.5456,2.5452,2.5456,2.5053,2.3877,2.5053,2.3877



=== half_width ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,1.9600,1.9602,1.9600,1.9623,2.1404,1.9779,2.1203
0.5000,-0.5000,1.9600,1.9602,1.9600,1.9623,2.1404,1.9779,2.1203
0.9000,0.9000,1.9600,1.9602,1.9600,1.9623,2.1404,1.9779,2.1203



=== coverage (target 0.95) ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,0.9500,0.9500,0.9500,0.9502,0.9593,0.9520,0.9573
0.5000,-0.5000,0.9500,0.9500,0.9500,0.9497,0.9436,0.9515,0.9412
0.9000,0.9000,0.9500,0.9500,0.9500,0.9501,0.9655,0.9519,0.9638


### 진짜 구간 $[L, U]$ 형태로 보기

앞의 표들은 center·half_width·coverage처럼 구간에서 계산한 숫자만 보여줬는데, 실제 구간 끝점을 `[L, U]` 문자열로 그대로 보여주는 표.

In [25]:
# L, U를 "[L, U]" 형태의 문자열로 합쳐서, family별로 나란히 볼 수 있는 표를 만듦
table_with_family["interval_str"] = table_with_family.apply(
    lambda row: f"[{row['L']:.4f}, {row['U']:.4f}]", axis=1
)

interval_bracket_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="interval_str")
interval_bracket_table = interval_bracket_table[FAMILY_ORDER]  # 열 순서를 5개 구간의 원래 순서로 고정
interval_bracket_table

,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,"[-1.9600, 1.9600]","[-1.9605, 1.9598]","[-1.9600, 1.9600]","[-1.9780, 1.9466]","[-2.4504, 1.8305]","[-1.9936, 1.9622]","[-2.4303, 1.8103]"
0.5000,-0.5000,"[-1.9600, 1.9600]","[-1.9605, 1.9598]","[-1.9600, 1.9600]","[-2.0335, 1.8911]","[-2.6593, 1.6215]","[-2.0491, 1.9067]","[-2.6392, 1.6014]"
0.9000,0.9000,"[0.5856, 4.5055]","[0.5851, 4.5054]","[0.5856, 4.5055]","[0.5429, 4.4676]","[0.2473, 4.5281]","[0.5273, 4.4832]","[0.2674, 4.5080]"


## 10. 지표의 4-component 분해 (structural / fitting / calibration / mc_approximation)

CLAUDE.md §6에 따라, 고정된 위치·반복·alpha에서 각 scalar metric $T$ (coverage, center, half_width, length)에 대해
다섯 구간(oracle, mc, res_true, pop, cp)의 값으로부터 다음 네 component와 total_mc를 계산한다.

| component | 정의 |
| --- | --- |
| `structural` | $T(C_{res,0}) - T(C_{Oracle})$ |
| `fitting` | $T(C_{pop,j}) - T(C_{res,0})$ |
| `calibration` | $T(C_{CP,j}) - T(C_{pop,j})$ |
| `mc_approximation` | $T(C_{Oracle}) - T(C_{MC})$ |
| `total_mc` | $T(C_{CP,j}) - T(C_{MC})$ |

`total_mc = structural + fitting + calibration + mc_approximation`은 telescoping 합이므로 대수적 항등식으로 항상 성립한다.
모든 분해는 **signed difference**이며 절댓값·제곱합으로 바꾸지 않는다 — 항끼리 상쇄될 수 있고, 독립적인 인과 기여율이 아니다.

Oracle·MC·res_true는 모델과 무관하므로 OLS·RF 간에 공유하고, pop·cp만 모델별로 따로 계산한다(§4).

In [26]:
def interval_metrics(L, U, X, dgp):
    '''구간 [L, U]에서 4개 scalar metric(coverage, center, half_width, length)을 계산.

    coverage는 CDF로 직접 계산(§6, §9: 이진 포함 횟수 추정으로 대체하지 않음).
    L, U, 반환값 모두 X와 같은 길이의 1차원 array.
    '''
    coverage = conditional_coverage(L, U, X, dgp)
    center = (L + U) / 2
    half_width = (U - L) / 2
    length = U - L
    return {"coverage": coverage, "center": center, "half_width": half_width, "length": length}

In [27]:
def decompose_metric(values):
    '''values: {"oracle","mc","res_true","pop","cp"} -> 같은 위치·반복·alpha에서의 T(C_family) 값(스칼라 또는 array).

    CLAUDE.md §6의 4개 component + total_mc를 signed difference로 계산해서 반환.
    `_identity_residual`은 total_mc - (4개 component 합)으로, telescoping 항등식이므로 항상 0에 가까워야 함
    (검증용으로만 사용하고 과학적 결과에는 포함하지 않음).
    '''
    structural = values["res_true"] - values["oracle"]
    fitting = values["pop"] - values["res_true"]
    calibration = values["cp"] - values["pop"]
    mc_approximation = values["oracle"] - values["mc"]
    total_mc = values["cp"] - values["mc"]

    identity_residual = total_mc - (structural + fitting + calibration + mc_approximation)

    return {
        "structural": structural,
        "fitting": fitting,
        "calibration": calibration,
        "mc_approximation": mc_approximation,
        "total_mc": total_mc,
        "_identity_residual": identity_residual,
    }


METRIC_NAMES = ["coverage", "center", "half_width", "length"]
COMPONENT_ORDER = ["structural", "fitting", "calibration", "mc_approximation", "total_mc"]

In [28]:
# dgp, alpha, x_eval, ols_model, rf_model, q0, q_pop_ols, q_pop_rf, q_hat_ols, q_hat_rf,
# eps_sample_gaussian은 위(§5~§9)에서 이미 계산해둔 값들을 그대로 재사용
# (linear_homo_gaussian, alpha=0.05, b=1)

# 모델과 무관한 3개 구간은 한 번만 계산해서 OLS·RF가 공유 (§4: 중복 생성 불필요)
shared_families = {
    "oracle": oracle_interval(x_eval, dgp, alpha),
    "mc": mc_interval(x_eval, dgp, alpha, eps_sample_gaussian),
    "res_true": res_true_interval(x_eval, dgp, alpha),
}
shared_metrics = {name: interval_metrics(L, U, x_eval, dgp) for name, (L, U) in shared_families.items()}

model_specific_families = {
    "OLS": {
        "pop": pop_interval(x_eval, ols_model, q_pop_ols),
        "cp": cp_interval(x_eval, ols_model, q_hat_ols),
    },
    "RF": {
        "pop": pop_interval(x_eval, rf_model, q_pop_rf),
        "cp": cp_interval(x_eval, rf_model, q_hat_rf),
    },
}

decomposition_rows = []
for model_id, model_families in model_specific_families.items():
    metrics_by_family = dict(shared_metrics)
    metrics_by_family.update(
        {name: interval_metrics(L, U, x_eval, dgp) for name, (L, U) in model_families.items()}
    )

    for metric_name in METRIC_NAMES:
        values = {family_name: metrics_by_family[family_name][metric_name] for family_name in
                  ["oracle", "mc", "res_true", "pop", "cp"]}
        components = decompose_metric(values)

        max_identity_residual = np.max(np.abs(components["_identity_residual"]))
        assert max_identity_residual < 1e-8, (
            f"{model_id}/{metric_name}: 분해 항등식 불일치(잔차={max_identity_residual:.3e})"
        )

        for point_idx in range(len(x_eval)):
            for component_name in COMPONENT_ORDER:
                decomposition_rows.append({
                    "model_id": model_id,
                    "x1": x_eval[point_idx, 0],
                    "x2": x_eval[point_idx, 1],
                    "metric": metric_name,
                    "component": component_name,
                    "value": components[component_name][point_idx],
                })

decomposition_table = pd.DataFrame(decomposition_rows)
print("모든 (model_id, metric)에서 total_mc = structural+fitting+calibration+mc_approximation 항등식 확인 완료 (|잔차| < 1e-8)")
decomposition_table.pivot(index=["model_id", "x1", "x2", "metric"], columns="component", values="value")[COMPONENT_ORDER]

모든 (model_id, metric)에서 total_mc = structural+fitting+calibration+mc_approximation 항등식 확인 완료 (|잔차| < 1e-8)


component                           structural  fitting  calibration  \
model_id x1     x2      metric                                         
OLS      0.0000 0.0000  center          0.0000  -0.0157       0.0000   
                        coverage       -0.0000   0.0002       0.0018   
                        half_width     -0.0000   0.0024       0.0156   
                        length         -0.0000   0.0048       0.0312   
         0.5000 -0.5000 center          0.0000  -0.0712       0.0000   
                        coverage       -0.0000  -0.0003       0.0018   
                        half_width     -0.0000   0.0024       0.0156   
                        length         -0.0000   0.0048       0.0312   
         0.9000 0.9000  center          0.0000  -0.0403       0.0000   
                        coverage       -0.0000   0.0001       0.0018   
                        half_width     -0.0000   0.0024       0.0156   
                        length         -0.0000   0.0048       0.0312   
RF       0.0000 0.0000  center          0.0000  -0.3100       0.0000   
                        coverage       -0.0000   0.0093      -0.0019   
                        half_width     -0.0000   0.1804      -0.0201   
                        length         -0.0000   0.3609      -0.0402   
         0.5000 -0.5000 center          0.0000  -0.5189       0.0000   
                        coverage       -0.0000  -0.0064      -0.0024   
                        half_width     -0.0000   0.1804      -0.0201   
                        length         -0.0000   0.3609      -0.0402   
         0.9000 0.9000  center          0.0000  -0.1579       0.0000   
                        coverage       -0.0000   0.0155      -0.0017   
                        half_width     -0.0000   0.1804      -0.0201   
                        length         -0.0000   0.3609      -0.0402   

component                           mc_approximation  total_mc  
model_id x1     x2      metric                                  
OLS      0.0000 0.0000  center                0.0004   -0.0153  
                        coverage             -0.0000    0.0020  
                        half_width           -0.0002    0.0178  
                        length               -0.0004    0.0356  
         0.5000 -0.5000 center                0.0004   -0.0708  
                        coverage             -0.0000    0.0015  
                        half_width           -0.0002    0.0178  
                        length               -0.0004    0.0356  
         0.9000 0.9000  center                0.0004   -0.0400  
                        coverage             -0.0000    0.0019  
                        half_width           -0.0002    0.0178  
                        length               -0.0004    0.0356  
RF       0.0000 0.0000  center                0.0004   -0.3096  
                        coverage             -0.0000    0.0073  
                        half_width           -0.0002    0.1601  
                        length               -0.0004    0.3203  
         0.5000 -0.5000 center                0.0004   -0.5185  
                        coverage             -0.0000   -0.0088  
                        half_width           -0.0002    0.1601  
                        length               -0.0004    0.3203  
         0.9000 0.9000  center                0.0004   -0.1575  
                        coverage             -0.0000    0.0138  
                        half_width           -0.0002    0.1601  
                        length               -0.0004    0.3203

### 중심·반폭 분해의 분석적 지름길 (§6 수식) 검증

$q_{lo}=F_\varepsilon^{-1}(\alpha/2)$, $q_{hi}=F_\varepsilon^{-1}(1-\alpha/2)$일 때 $a_\alpha=(q_{lo}+q_{hi})/2$, $h_\alpha=(q_{hi}-q_{lo})/2$로 두면 CLAUDE.md §6은 다음 닫힌형태를 제시한다.

- 중심 분해: `(structural, fitting, calibration) = (-sigma0(x)*a_alpha, fitted_mean(x) - true_mean(x), 0)`
- 반폭 분해: `(structural, fitting, calibration) = (q0 - sigma0(x)*h_alpha, q_pop - q0, q_hat - q_pop)`

위 §10에서 다섯 구간의 실제 endpoint로부터 직접 계산한 값과 이 닫힌형태 공식이 일치하는지 확인한다.
(길이 분해는 반폭 분해의 2배이므로 별도로 검증하지 않음.)

In [29]:
q_lo_analytic = dgp.error.ppf(alpha / 2)
q_hi_analytic = dgp.error.ppf(1 - alpha / 2)
a_alpha = (q_lo_analytic + q_hi_analytic) / 2
h_alpha = (q_hi_analytic - q_lo_analytic) / 2

sigma_at_eval = dgp.scale_fn(x_eval)
true_mean_at_eval = dgp.mean_fn(x_eval)

for model_id, fitted_model, q_pop, q_hat in [
    ("OLS", ols_model, q_pop_ols, q_hat_ols),
    ("RF", rf_model, q_pop_rf, q_hat_rf),
]:
    fitted_mean_at_eval = fitted_model.predict(x_eval)

    analytic_values = {
        ("center", "structural"): -sigma_at_eval * a_alpha,
        ("center", "fitting"): fitted_mean_at_eval - true_mean_at_eval,
        ("center", "calibration"): np.zeros(len(x_eval)),
        ("half_width", "structural"): q0 - sigma_at_eval * h_alpha,
        ("half_width", "fitting"): q_pop - q0,
        ("half_width", "calibration"): q_hat - q_pop,
    }

    for (metric_name, component_name), analytic_value in analytic_values.items():
        direct_value = decomposition_table.loc[
            (decomposition_table["model_id"] == model_id)
            & (decomposition_table["metric"] == metric_name)
            & (decomposition_table["component"] == component_name),
            "value",
        ].to_numpy()
        max_gap = np.max(np.abs(direct_value - analytic_value))
        print(f"{model_id:>3} {metric_name:>10} {component_name:>12}: max|direct-analytic| = {max_gap:.2e}")
        assert max_gap < 1e-8, f"{model_id}/{metric_name}/{component_name}: 분석적 지름길과 직접 계산 불일치"

print("\n중심·반폭 분해의 분석적 지름길(§6 수식)과 직접 계산 일치 확인 완료")

OLS     center   structural: max|direct-analytic| = 2.22e-16
OLS     center      fitting: max|direct-analytic| = 1.39e-17
OLS     center  calibration: max|direct-analytic| = 1.11e-16
OLS half_width   structural: max|direct-analytic| = 2.22e-16
OLS half_width      fitting: max|direct-analytic| = 0.00e+00
OLS half_width  calibration: max|direct-analytic| = 2.22e-16
 RF     center   structural: max|direct-analytic| = 2.22e-16
 RF     center      fitting: max|direct-analytic| = 1.11e-16
 RF     center  calibration: max|direct-analytic| = 0.00e+00
 RF half_width   structural: max|direct-analytic| = 2.22e-16
 RF half_width      fitting: max|direct-analytic| = 2.22e-16
 RF half_width  calibration: max|direct-analytic| = 0.00e+00

중심·반폭 분해의 분석적 지름길(§6 수식)과 직접 계산 일치 확인 완료
